# 02 — Basic image analysis

Use this for cleaned image previews, simple source finding, aperture flux estimates, and radial profiles.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astro_utils import (
    read_frame_cube,
    read_fits_data,
    background_subtract,
    robust_limits,
    show_image,
    centroid_bright_source,
    circular_aperture_flux,
)


In [ ]:
FITS_PATH = Path("data/your_file.fits")
HDU_NAME = "SCI"
USE_MEDIAN_FRAME = True
APERTURE_RADIUS = 15


In [ ]:
data, header = read_fits_data(FITS_PATH, ext=HDU_NAME)

if data.ndim == 2:
    img = np.asarray(data, dtype=float)
else:
    cube, _ = read_frame_cube(FITS_PATH, ext=HDU_NAME)
    img = np.nanmedian(cube, axis=0) if USE_MEDIAN_FRAME else cube[0]

img_bs = background_subtract(img)
show_image(img_bs, title="Background-subtracted image")


In [ ]:
x0, y0 = centroid_bright_source(img_bs)
flux, mask = circular_aperture_flux(img_bs, x0=x0, y0=y0, r=APERTURE_RADIUS)

print(f"Estimated centroid: x={x0:.2f}, y={y0:.2f}")
print(f"Aperture flux (r={APERTURE_RADIUS}): {flux:.3f}")


In [ ]:
plt.figure(figsize=(7, 7))
vmin, vmax = robust_limits(img_bs)
plt.imshow(img_bs, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
yy, xx = np.where(mask)
plt.scatter([x0], [y0], s=40, marker="x")
plt.contour(mask.astype(float), levels=[0.5], origin="lower")
plt.title("Detected bright source and aperture")
plt.tight_layout()
plt.show()


In [ ]:
# Radial profile around the centroid
y, x = np.indices(img_bs.shape)
r = np.sqrt((x - x0)**2 + (y - y0)**2)
r_int = r.astype(int)

radial_mean = np.array([np.nanmean(img_bs[r_int == k]) for k in range(r_int.max() + 1)])

plt.figure(figsize=(8, 4))
plt.plot(radial_mean)
plt.xlabel("Radius (pixels)")
plt.ylabel("Mean signal")
plt.title("Radial profile")
plt.tight_layout()
plt.show()


### Extend this notebook
- Replace the centroid function with a catalog position
- Add annulus background subtraction
- Add WCS overlays if the header has sky coordinates
- Save source measurements to CSV
